# Power Dataset Analysis For
# Self-Supervised Learning for Cascading Failure Prediction


Author: Minglei Zhou

## 1. Introduction

Cascading failures are a serious problem in power systems.

A small disturbance can spread through the network.

This may lead to large-scale blackout.

Most existing methods need labeled data.

But labeled data is very limited.

So we need a new method.

## 2. Research Goal

My goal is to use self-supervised learning.

We learn from large unlabeled data.

Then we apply it to prediction tasks.
Two main tasks:

1. Early warning (classification)

2. Risk assessment (regression)

## 3. Data

This research uses three public data sources.

1. **PGLib-OPF**

2. **OPFData**

3. **PowerGraph**


### 3.1 Summary Table

| Dataset | Main Role | Why It Is Important |
|---------|-----------|---------------------|
| PGLib-OPF | Network template | Gives buses, lines, and electrical structure |
| OPFData | Pretraining data | Gives many unlabeled operational samples |
| PowerGraph | Downstream benchmark | Gives graph tasks for prediction |

### 3.2 logic:

- PGLib-OPF → graph structure

- OPFData → unlabeled operational states

- PowerGraph → prediction tasks

### 3.4 File Structure

In [4]:
from pathlib import Path
import np
BASE = Path("power_demo_work")
for p in BASE.iterdir():
    print(p)


AttributeError: `np.float_` was removed in the NumPy 2.0 release. Use `np.float64` instead.

### 3.5 PGLib-OPF

PGLib-OPF to understand the physical structure of the power grid.

basic network template.
It includes:

- bus information
- branch information
- electrical parameters

This part helps me build the graph structure.

In [ ]:
import re
import pandas as pd
from IPython.display import display

pglib_file = BASE / "pglib" / "pglib_opf_case118_ieee.m"
text = pglib_file.read_text()

def parse_block(name):
    m = re.search(rf"{name}\s*=\s*\[(.*?)\];", text, re.S)
    rows = []
    for line in m.group(1).splitlines():
        line = line.strip()
        if not line or line.startswith("%"):
            continue
        line = line.split("%")[0]
        rows.append([float(x) for x in line.replace(";", "").split()])
    return rows

bus = parse_block("mpc.bus")
branch = parse_block("mpc.branch")

print("Number of buses:", len(bus))
print("Number of branches:", len(branch))

In [ ]:
bus_df = pd.DataFrame(bus)
bus_df.columns = [
    "bus_id", "bus_type", "Pd", "Qd",
    "Gs", "Bs", "area", "Vm", "Va",
    "baseKV", "zone", "Vmax", "Vmin"
]

branch_df = pd.DataFrame(branch)
branch_df.columns = [
    "from_bus", "to_bus", "r", "x", "b",
    "rateA", "rateB", "rateC",
    "ratio", "angle", "status",
    "angmin", "angmax"
]

print("Bus table:")
display(bus_df.head())

print("Branch table:")
display(branch_df.head())

The static physical structure.
·
- **bus** means node in the grid
- **branch** means line connection

- **Pd / Qd** mean demand
- **r / x / b** are line parameters

In [ ]:
print("Bus summary:")
display(bus_df.describe())
print("Branch summary:")
display(branch_df.describe())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(bus_df["Pd"], bins=20)
plt.title("Active Power Demand Distribution (Pd)")
plt.xlabel("Pd")
plt.ylabel("Count")
plt.show()

plt.hist(branch_df["r"], bins=20)
plt.title("Line Resistance Distribution")
plt.xlabel("r")
plt.ylabel("Count")
plt.show()

In [ ]:
import networkx as nx

G = nx.Graph()

for _, row in bus_df.iterrows():
    G.add_node(int(row["bus_id"] - 1), Pd=row["Pd"])

for _, row in branch_df.iterrows():
    G.add_edge(
        int(row["from_bus"] - 1),
        int(row["to_bus"] - 1),
        r=row["r"],
        x=row["x"]
    )

pos = nx.spring_layout(G, seed=42)

plt.figure(figsize=(8, 6))
nx.draw(G, pos, node_size=80, with_labels=False)
plt.title("PGLib Network Topology")
plt.show()

This figure shows the grid as a graph.

Nodes are buses, and edges are transmission lines.


This is the first step of graph-based modeling.


### 3.6 OPFData

operational states.

This dataset contains many operating samples under different conditions.

It is important for self-supervised learning.

Because it provides large unlabeled data.

For my research, OPFData is mainly used for representation pretraining.

In [ ]:
import tarfile
opf_tar = BASE / "opfdata" / "dataset_release_1__pglib_opf_case14_ieee_0.tar.gz"
extract_dir = BASE / "opfdata" / "case14_extracted"

if not extract_dir.exists():
    print("Extracting OPFData...")
    with tarfile.open(opf_tar, "r:gz") as tf:
        tf.extractall(extract_dir)

print("Done:", extract_dir)

example_files = sorted(extract_dir.rglob("example_*.json"))
print("num example files found:", len(example_files))
print("first 5 files:")
for p in example_files[:5]:
    print(" -", p)

In [ ]:
import json

sample_path = example_files[0]
print("Using sample file:", sample_path)

with open(sample_path, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("Top-level keys:", list(sample.keys()))

In [ ]:
grid = sample["grid"]
solution = sample["solution"]
metadata = sample["metadata"]

print("Grid keys:", list(grid.keys()))
print("Solution keys:", list(solution.keys()))
print("Metadata keys:", list(metadata.keys()))

The sample has three main parts:

- `grid`: static network structure

- `solution`: dynamic operating results

- `metadata`: additional sample information

I need both topology and operating state.

A useful point in OPFData is that it separates static and dynamic information.

This is very suitable for graph-based learning.

Because the graph structure is relatively stable,

while operating states change across samples.

In [ ]:
print("Grid node keys:", list(grid["nodes"].keys()))
print("Solution node keys:", list(solution["nodes"].keys()))

In [ ]:
import pandas as pd
from IPython.display import display

grid_bus_df = pd.DataFrame(grid["nodes"]["bus"])
grid_bus_df.columns = ["col_0", "bus_type", "v_min", "v_max"]

grid_bus_df = grid_bus_df.reset_index(drop=True)
grid_bus_df["bus_index"] = grid_bus_df.index + 1

print("Static bus table:")
display(grid_bus_df.head())

In [ ]:
solution_bus_df = pd.DataFrame(solution["nodes"]["bus"])
solution_bus_df.columns = ["Va", "Vm"]

solution_bus_df = solution_bus_df.reset_index(drop=True)
solution_bus_df["bus_index"] = solution_bus_df.index + 1

print("Dynamic bus table:")
display(solution_bus_df.head())


- `Va`: voltage angle

- `Vm`: voltage magnitude

In [ ]:
bus_full_df = pd.concat(
    [
        grid_bus_df[["bus_index", "bus_type", "v_min", "v_max"]],
        solution_bus_df[["Va", "Vm"]]
    ],
    axis=1
)

print("Combined bus table:")
display(bus_full_df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

x = bus_full_df["bus_index"]

plt.plot(x, bus_full_df["Vm"], marker="o", linewidth=2, label="Actual Voltage (Vm)")
plt.axhline(bus_full_df["v_min"].iloc[0], linestyle="--", label="Lower Limit")
plt.axhline(bus_full_df["v_max"].iloc[0], linestyle="--", label="Upper Limit")

plt.title("Bus Voltage: Actual vs Limits")
plt.xlabel("Bus Index")
plt.ylabel("Voltage (p.u.)")
plt.xticks(x)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

The dashed lines show the allowed voltage range.

The solid line shows the actual voltage magnitude at each bus.

We can see that the voltage changes across buses,

while still staying within the limits in this sample.

This dynamic information is much more useful for representation learning.

## 3.7 PowerGraph


PowerGraph is used for downstream graph learning tasks.
It is important because it is closer to my final prediction goal.
In my proposal, PowerGraph supports tasks such as early warning and risk assessment.

In [ ]:
raw = BASE / "powergraph" / "dataset_cascades_extracted" / "dataset_cascades" / "ieee24" / "ieee24" / "raw"

mat_files = sorted(raw.glob("*.mat"))
print("MAT files:")
for p in mat_files:
    print(" -", p.name)

def inspect_h5mat(path):
    rows = []

    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            rows.append({
                "name": name,
                "shape": obj.shape,
                "dtype": str(obj.dtype)
            })

    with h5py.File(path, "r") as f:
        top_keys = list(f.keys())

    with h5py.File(path, "r") as f:
        f.visititems(visitor)

    return top_keys, pd.DataFrame(rows)

summary_tables = {}

for p in mat_files:
    print(f"\n===== {p.name} =====")
    top_keys, df_info = inspect_h5mat(p)
    summary_tables[p.name] = df_info
    print("top-level keys:", top_keys)
    display(df_info.head(20))

PowerGraph is stored in MATLAB `.mat` files.
（PowerGraph 以 MATLAB 的 .mat 文件形式存储）

node features, edge features, topology, and labels are stored.
（这帮助我理解节点特征、边特征、拓扑和标签分别存在哪里）

In [ ]:
import h5py

def read_h5_ref_array(mat_path, main_key):
    out = []
    with h5py.File(mat_path, "r") as f:
        arr = f[main_key]
        refs = np.array(arr).flatten()
        for ref in refs:
            obj = f[ref]
            out.append(np.array(obj))
    return out

def read_h5_numeric(mat_path, key):
    with h5py.File(mat_path, "r") as f:
        return np.array(f[key])

# ---------- file paths ----------
bf_path = raw / "Bf.mat"
ef_path = raw / "Ef.mat"
blist_path = raw / "blist.mat"
of_bi_path = raw / "of_bi.mat"
of_mc_path = raw / "of_mc.mat"
of_reg_path = raw / "of_reg.mat"

# ---------- load arrays ----------
B_list = read_h5_ref_array(bf_path, "B_f_tot")        # each item: (3, 24)
E_list = read_h5_ref_array(ef_path, "E_f_post")       # each item: (4, 38)

bList = read_h5_numeric(blist_path, "bList")          # (2, 38)
y_bi_all = read_h5_ref_array(of_bi_path, "output_features")  # each item: (1,1)
y_mc_all = read_h5_ref_array(of_mc_path, "category")         # each item: (4,1)
y_reg_all = read_h5_numeric(of_reg_path, "dns_MW")           # (1, 21500)

print("Number of graph samples in B_list:", len(B_list))
print("Number of graph samples in E_list:", len(E_list))
print("bList shape:", bList.shape)
print("y_reg_all shape:", y_reg_all.shape)

# ---------- pick one sample ----------
sample_idx = 0

B = B_list[sample_idx]   # (3,24)
E = E_list[sample_idx]   # (4,38)

node_feat = B.T          # (24,3)
edge_feat = E.T          # (38,4)
edge_index = bList.T     # (38,2)

node_df = pd.DataFrame(node_feat, columns=["node_f0", "node_f1", "node_f2"])
node_df["node_id"] = np.arange(len(node_df))
node_df = node_df[["node_id", "node_f0", "node_f1", "node_f2"]]

edge_df = pd.DataFrame(edge_index, columns=["src", "dst"])
edge_feat_df = pd.DataFrame(edge_feat, columns=["edge_f0", "edge_f1", "edge_f2", "edge_f3"])
edge_df = pd.concat([edge_df, edge_feat_df], axis=1)

# labels
y_bi = float(np.array(y_bi_all[sample_idx]).squeeze())
y_mc = np.array(y_mc_all[sample_idx]).squeeze()
y_reg = float(np.array(y_reg_all).squeeze()[sample_idx])

label_df = pd.DataFrame([{
    "sample_idx": sample_idx,
    "binary_label": y_bi,
    "multiclass_raw": y_mc.tolist() if hasattr(y_mc, "tolist") else y_mc,
    "regression_label": y_reg
}])

print("Node table:")
display(node_df.head())

print("Edge table:")
display(edge_df.head())

print("Label table:")
display(label_df)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

plot_edge_df = edge_df.copy()

# 如果边是 1-based，就转成 0-based；如果已经是 0-based，也不会影响展示太多
if plot_edge_df["src"].min() >= 1 and plot_edge_df["dst"].min() >= 1:
    plot_edge_df["src_plot"] = plot_edge_df["src"].astype(int) - 1
    plot_edge_df["dst_plot"] = plot_edge_df["dst"].astype(int) - 1
else:
    plot_edge_df["src_plot"] = plot_edge_df["src"].astype(int)
    plot_edge_df["dst_plot"] = plot_edge_df["dst"].astype(int)

# ---------- 2. 构建图 ----------
G_pg = nx.Graph()

# 节点
for _, row in node_df.iterrows():
    nid = int(row["node_id"])
    G_pg.add_node(
        nid,
        node_f0=row["node_f0"],
        node_f1=row["node_f1"],
        node_f2=row["node_f2"],
    )

# 边
for _, row in plot_edge_df.iterrows():
    G_pg.add_edge(
        int(row["src_plot"]),
        int(row["dst_plot"]),
        edge_f0=row["edge_f0"],
        edge_f1=row["edge_f1"],
        edge_f2=row["edge_f2"],
        edge_f3=row["edge_f3"],
    )

print("Nodes:", G_pg.number_of_nodes())
print("Edges:", G_pg.number_of_edges())

# ---------- 3. 可视化属性 ----------
node_colors = [G_pg.nodes[n]["node_f0"] for n in G_pg.nodes()]
node_sizes = [abs(G_pg.nodes[n]["node_f1"]) * 800 + 120 for n in G_pg.nodes()]
edge_widths = [abs(G_pg[u][v]["edge_f0"]) * 4 + 0.5 for u, v in G_pg.edges()]

# 固定布局，方便重复展示
pos_pg = nx.spring_layout(G_pg, seed=42)

# ---------- 4. 绘图 ----------
plt.figure(figsize=(8, 6))

nx.draw_networkx_edges(
    G_pg,
    pos_pg,
    width=edge_widths,
    alpha=0.6
)

nodes = nx.draw_networkx_nodes(
    G_pg,
    pos_pg,
    node_color=node_colors,
    node_size=node_sizes,
    cmap=plt.cm.viridis
)

nx.draw_networkx_labels(
    G_pg,
    pos_pg,
    font_size=8
)

plt.colorbar(nodes, label="node_f0")
plt.title(
    f"PowerGraph Sample {sample_idx}\n"
    f"binary={y_bi}, regression={y_reg:.2f}, multiclass={y_mc.tolist()}"
)
plt.axis("off")
plt.show()

This figure shows one sample from PowerGraph.

It contains 24 nodes and 34 edges.
So each sample can be viewed as a graph.

The node color represents one node feature.
The node size represents another node feature.
The edge width represents one edge feature.

We can also see that this sample has graph-level labels,
including binary, regression, and multiclass targets.

So this dataset is not only a graph structure,
but also a complete graph learning sample for downstream tasks.

### Current Problems

Now I use three datasets: PGLib, OPFData, and PowerGraph.

But they are not fully connected.

They do not come from the same data source


Their features are different

So this is not a full data loop.

### Possible Solution

I need build a data-level closed loop using OPFData.


Structure + state + label are unified



# 4. Training Method

1. Graph encoder

2. Self-supervised learning module

3. Downstream prediction model

### 5.2 Graph Encoder



It takes a graph as input.


Each graph has:

- node features

- edge features

- graph structure

The encoder outputs node embeddings.

These embeddings capture the structure and state of the system.

### 5.2 Self-Supervised Model: GraphMAE
GraphMAE is useful because it can learn from large unlabeled data and produce general graph representations.
| Aspect | GraphMAE | Traditional Supervised Learning |
|--------|----------|--------------------------------|
| Data requirement | Does not need labels | Needs labeled data |
| Data usage | Can use large unlabeled OPFData | Limited by labeled data |
| Learning ability | Learns general graph patterns | Learns task-specific patterns |
| Robustness | More robust to noise and missing data | Sensitive to data quality |
| Transferability | Can transfer to new datasets | Hard to transfer |
| Suitability | Good for pretraining stage | Good for final task only |

### Alternatives to GraphMAE

| Method | Idea | Advantage | Limitation |
|--------|------|----------|------------|
| GraphMAE | Mask and reconstruct node features | Simple, good for continuous data | Ignores contrastive signals |
| GRACE | Contrastive learning (graph augmentations) | Strong representation learning | Needs careful augmentation |
| DGI | Maximize mutual information | Simple and efficient | Less powerful on complex graphs |
| BGRL | Bootstrap without negative samples | Stable training | More complex structure |
| GraphCL | Contrastive learning with augmentations | Good generalization | Sensitive to augmentation design |

#  Evaluation

### Evaluation 1: Prediction Performance

| Task Type | Metrics | Purpose |
|-----------|--------|--------|
| Classification | Accuracy, F1-score, AUC | Measure how well the model predicts risk classes |
| Regression | MAE, RMSE | Measure how close predictions are to true values |

### Evaluation 2: Transfer

| Evaluation | Method | Purpose |
|------------|--------|--------|
| Transfer learning | Compare with / without pretraining | Check if OPFData pretraining is useful |
| Generalization | Test on PowerGraph dataset | Check if model works on new data |